# Job 0: import diffuse map from mdx2

In [1]:
% calculated in mdx2 by scaling the Bragg intensities to Fobs^2 column in import/hklout.mtz
scale_factor = 0.0019538503

unit_cell = h5read('import/geometry.nxs','/entry/crystal/unit_cell');
space_group_number = h5read('import/geometry.nxs','/entry/symmetry/space_group_number');
ndiv = h5read('import/merged_5_5_11.nxs','/entry/hkl_table/ndiv');
ndiv = double(ndiv'); % convert to native matlab type from int64
c = num2cell(unit_cell); % temporary copy to unpack as function args in latt.Basis
MT = proc.script.MapTools(...
    'SpaceGroup',symm.SpaceGroup(space_group_number),...
    'Basis',latt.Basis(c{:}).invert,...
    'Grid',latt.PeriodicGrid(ndiv,[0,0,0],[1,1,1]),...
    'type','intensity');


scale_factor =

    0.0020



In [2]:
h = h5read('import/merged_5_5_11.nxs','/entry/hkl_table/h');
k = h5read('import/merged_5_5_11.nxs','/entry/hkl_table/k');
l = h5read('import/merged_5_5_11.nxs','/entry/hkl_table/l');
I = scale_factor * h5read('import/merged_5_5_11.nxs','/entry/hkl_table/intensity');
sigma = scale_factor * h5read('import/merged_5_5_11.nxs','/entry/hkl_table/intensity_error');
hkl_table = table(h,k,l,I,sigma);
disp([min(h),max(h),min(k),max(k),min(l),max(l)])
clearvars h k l I sigma

         0   84.8000         0   86.0000         0   35.6364



In [3]:
MT = MT.resize('roi',[-2/5,86 + 2/5,-2/5,86 + 2/5,-5/11,36 + 5/11]); % generous padding

In [4]:
[I,sigma] = MT.table2array(hkl_table,'direct','replace',NaN);

mode = direct, mode2 = replace, default value = NaN


In [5]:
% apply high resolution cutoff
smax = 0.8;

msk = MT.spherical_mask(smax);
I(~msk) = NaN;
sigma(~msk) = NaN;
clearvars msk

In [6]:
MT.export('export/mac1_5x5x11_map.h5','total',I,sigma,'Datatype','single','ChunkSize',ndiv);

> In proc.script/MapTools/export (line 319)
Creating dataset in export/mac1_5x5x11_map.h5
	Location: /maps/total/I
	Options: Columns 1 through 4

    {[435 435 407], 'Datatype', 'single', 'ChunkSize'}

  Columns 5 through 7

    {[5 5 11], 'Deflate', [1]}
Writing data to export/mac1_5x5x11_map.h5
	Location: /maps/total/I
	Options: {}
> In H5D.write (line 100)
In h5write (line 175)
In io.h5/FileInterface/write (line 74)
In io.h5/MapFile/new_dataset (line 37)
In proc.script/MapTools/export (line 340)
Done.
Creating dataset in export/mac1_5x5x11_map.h5
	Location: /maps/total/sigma
	Options: Columns 1 through 4

    {[435 435 407], 'Datatype', 'single', 'ChunkSize'}

  Columns 5 through 7

    {[5 5 11], 'Deflate', [1]}
Writing data to export/mac1_5x5x11_map.h5
	Location: /maps/total/sigma
	Options: {}
> In H5D.write (line 100)
In h5write (line 175)
In io.h5/FileInterface/write (line 74)
In io.h5/MapFile/new_dataset (line 37)
In proc.script/MapTools/export (line 340)
Done.
